# Taller 5 — Evaluación de LLMs

Notebook con **código para cada ejercicio** de las slides `_taller_5_LLMs_evaluation.tex`.

Cubre: comparación de modelos, métricas clásicas (BLEU vs. semántica), contaminación de benchmarks, LLM-as-a-judge, **DeepEval**, **RAGAS**, **LangSmith** y **evaluación de agentes** (tool correctness).

## Requisitos / API keys

Configura un archivo `.env` (misma carpeta) con las llaves que vayas a usar:

- `OPENAI_API_KEY` — modelos OpenAI, embeddings y juez por defecto de DeepEval/RAGAS.
- `ANTHROPIC_API_KEY` — modelos Claude.
- `GEMINI_API_KEY` — modelos Gemini.
- `LANGSMITH_API_KEY`, `LANGSMITH_TRACING=true` — para el ejercicio de LangSmith.

> No necesitas todas: cada sección indica qué llave requiere.

In [ ]:
# Instala dependencias (descomenta si hace falta)
# !pip install -U langchain langchain-openai langchain-anthropic langchain-google-genai \
#     dotenv nltk sacrebleu numpy deepeval ragas langsmith datasets

In [5]:
import os
import json
import numpy as np
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

from langchain.chat_models import init_chat_model

# Modelos que usaremos (ajusta segun las llaves que tengas)
MODELS = {
    "gpt":    "openai:gpt-5.4-mini",
    "claude": "anthropic:claude-haiku-4-5",
    "gemini": "google_genai:gemini-2.5-flash",
}

def ask(model_key, prompt, **kw):
    """Envia un prompt a un modelo y devuelve el texto de la respuesta."""
    llm = init_chat_model(MODELS[model_key], **kw)
    return llm.invoke(prompt).content

---
## Ejercicio 0 — Comparar 3 modelos

Pregunta lo mismo a ChatGPT, Gemini y Claude y compara las respuestas.

In [6]:
PREGUNTA = (
    "Lista 3 limitaciones de los benchmarks de LLMs "
    "y explica brevemente cada una."
)

for key in MODELS:
    try:
        resp = ask(key, PREGUNTA)
        display(Markdown(f"### {key}\n{resp}"))
    except Exception as e:
        print(f"[{key}] no disponible: {e}")

### gpt
Claro. Aquí van **3 limitaciones de los benchmarks de LLMs**:

1. **No siempre reflejan el uso real**  
   Muchos benchmarks evalúan tareas muy específicas o artificiales, que no representan cómo se usan los modelos en situaciones reales, como conversaciones largas, trabajo creativo o resolución de problemas abiertos.

2. **Se pueden “sobreajustar” a los benchmarks**  
   Los modelos y sus ajustes pueden optimizarse para rendir bien en pruebas concretas sin mejorar realmente su capacidad general. Esto puede dar una imagen exagerada de su rendimiento.

3. **Pueden estar desactualizados o tener sesgos**  
   Algunos benchmarks usan datos antiguos o mal distribuidos, por lo que no capturan conocimientos recientes ni la diversidad de idiomas, culturas o contextos. Esto puede favorecer a ciertos modelos sobre otros.

Si quieres, también puedo darte estas limitaciones en formato **más técnico**, **más breve**, o con **ejemplos concretos**.

### claude
# 3 Limitaciones de los Benchmarks de LLMs

## 1. **Desalineación con el desempeño real**
Los benchmarks tradicionales miden capacidades específicas en entornos controlados, pero no siempre reflejan cómo los modelos funcionan en aplicaciones del mundo real. Un LLM puede obtener puntuaciones altas en pruebas de razonamiento lógico pero fallar en tareas prácticas que requieren contexto y matices.

## 2. **Contaminación por datos de entrenamiento**
Es difícil garantizar que los benchmarks no estén incluidos en los datos de entrenamiento del modelo. Esto puede inflar artificialmente las puntuaciones, haciendo parecer que el modelo es más capaz de lo que realmente es cuando se enfrenta a problemas completamente nuevos.

## 3. **Falta de evaluación de habilidades emergentes**
Los benchmarks están diseñados antes de que se desarrollen los modelos, por lo que no pueden medir capacidades inesperadas o comportamientos que emergen en sistemas grandes. Esto deja fuera aspectos importantes de la inteligencia del modelo que no fueron anticipados.

---

**Consecuencia:** La confianza excesiva en benchmarks puede llevar a evaluaciones engañosas de los LLMs y decisiones deficientes sobre su implementación.

### gemini
Aquí tienes 3 limitaciones comunes de los benchmarks de LLMs, explicadas brevemente:

1.  **Contaminación de Datos y Memorización:**
    *   **Explicación:** Muchos LLMs son entrenados con vastas cantidades de datos de internet. Existe un riesgo significativo de que los conjuntos de datos utilizados para los benchmarks ya hayan sido incluidos, total o parcialmente, en los datos de entrenamiento de los modelos. Esto permite que el LLM "memorice" las respuestas en lugar de realmente razonar o generalizar, inflando artificialmente las puntuaciones y dando una falsa impresión de sus capacidades reales.

2.  **Falta de Aplicabilidad en el Mundo Real y Nuance:**
    *   **Explicación:** Los benchmarks a menudo consisten en tareas simplificadas, descontextualizadas o de opción múltiple que no reflejan la complejidad y la naturaleza abierta de las interacciones y aplicaciones del mundo real. Un alto rendimiento en un benchmark específico no garantiza que el modelo funcionará bien en escenarios prácticos que requieren creatividad, razonamiento de sentido común, manejo de la ambigüedad, interacción conversacional prolongada o comprensión de la intención del usuario.

3.  **Cobertura Limitada de Capacidades Clave:**
    *   **Explicación:** Los benchmarks actuales tienden a enfocarse en habilidades fácilmente cuantificables como la respuesta a preguntas, el resumen o la traducción. Sin embargo, luchan por evaluar aspectos cruciales y más complejos de la inteligencia de un LLM, como la creatividad genuina, el razonamiento ético, la seguridad (evitar contenido dañino), la coherencia a largo plazo en una conversación, la capacidad de detectar y corregir errores, o la comprensión profunda del contexto y el matiz emocional. Esto deja una imagen incompleta de las verdaderas capacidades y limitaciones del modelo.

---
## Ejercicio 1 — BLEU vs. similitud semántica

Dos frases con el **mismo significado** pero distintas palabras: BLEU las penaliza (poco solapamiento de n-gramas) mientras que la similitud semántica (embeddings) las reconoce como cercanas.

In [9]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt', quiet=True)

ref  = "El gato esta sobre la alfombra"
cand = "Sobre el tapete se encuentra el felino"

# --- BLEU (basado en solapamiento de n-gramas) ---
smooth = SmoothingFunction().method1
bleu = sentence_bleu([ref.lower().split()], cand.lower().split(),
                     smoothing_function=smooth)
print(f"BLEU: {bleu:.3f}  (bajo: casi no comparten palabras)")

BLEU: 0.039  (bajo: casi no comparten palabras)


In [10]:
# --- Similitud semantica con embeddings de OpenAI ---
from langchain_openai import OpenAIEmbeddings

emb = OpenAIEmbeddings(model="text-embedding-3-small")
v1, v2 = emb.embed_query(ref), emb.embed_query(cand)

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Similitud semantica (coseno): {cosine(v1, v2):.3f}  (alta: mismo significado)")

Similitud semantica (coseno): 0.753  (alta: mismo significado)


---
## Ejercicio 2 — Reto de contaminación (MMLU)

Tomamos una pregunta tipo MMLU y una **variante** con la misma lógica pero distintos valores. Si el modelo acierta la original pero falla la variante, es señal de **memorización**.

In [13]:
original = (
    "Pregunta de fisica. Un bloque de 2 kg cae desde 5 m sin friccion. "
    "Velocidad aproximada al llegar a la base (g=9.8)? "
    "A) 5.0  B) 7.0  C) 9.9  D) 14.0 m/s. Responde solo la letra."
)
variante = (
    "Pregunta de fisica. Un bloque de 3 kg cae desde 20 m sin friccion. "
    "Velocidad aproximada al llegar a la base (g=9.8)? "
    "A) 14.0  B) 19.8  C) 25.0  D) 40.0 m/s. Responde solo la letra."
)

print("Original  ->", ask("gemini", original))   # esperado: C (9.9)
print("Variante  ->", ask("gemini", variante))   # esperado: B (19.8)

Original  -> C
Variante  -> B)


---
## Ejercicio 4 — LLM-as-a-judge y sesgo de posición

Un LLM juez compara dos respuestas. Ejecutamos **dos veces intercambiando el orden A/B**: si el veredicto cambia, hay sesgo de posición.

In [14]:
JUEZ = """Eres un evaluador experto e imparcial. Compara las dos respuestas
a la PREGUNTA segun: (1) correctitud factual, (2) utilidad, (3) claridad.

PREGUNTA: {pregunta}
RESPUESTA A: {a}
RESPUESTA B: {b}

Razona en una linea y termina con: VEREDICTO: A | B | EMPATE"""

pregunta = "Explica que es la perplexity en una frase."
r1 = "Es una metrica que mide cuan sorprendido esta el modelo ante un texto; menor es mejor."
r2 = "La perplexity es el exponencial de la entropia cruzada promedio por token."

v_ab = ask("claude", JUEZ.format(pregunta=pregunta, a=r1, b=r2))
v_ba = ask("claude", JUEZ.format(pregunta=pregunta, a=r2, b=r1))  # orden invertido
print("Orden A=r1,B=r2 ->", v_ab.splitlines()[-1])
print("Orden A=r2,B=r1 ->", v_ba.splitlines()[-1])

Orden A=r1,B=r2 -> **VEREDICTO: A**
Orden A=r2,B=r1 -> **VEREDICTO: B**


---
## Ejercicio 5 — DeepEval (answer relevancy, faithfulness, hallucination)

DeepEval evalúa casos como tests. Por defecto usa un LLM de OpenAI como juez (requiere `OPENAI_API_KEY`). Creamos 3 casos: correcto, con alucinación e irrelevante.

In [16]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric, FaithfulnessMetric, HallucinationMetric,
)

contexto = ["Lima es la capital y la ciudad mas grande de Peru."]

caso_ok = LLMTestCase(
    input="Cual es la capital de Peru?",
    actual_output="La capital de Peru es Lima.",
    retrieval_context=contexto, context=contexto)

caso_alucina = LLMTestCase(
    input="Cual es la capital de Peru?",
    actual_output="La capital de Peru es Cusco, fundada en 1990.",
    retrieval_context=contexto, context=contexto)

caso_irrelevante = LLMTestCase(
    input="Cual es la capital de Peru?",
    actual_output="El ceviche es un plato tipico muy popular.",
    retrieval_context=contexto, context=contexto)

In [17]:
for nombre, caso in [("OK", caso_ok), ("ALUCINA", caso_alucina),
                     ("IRRELEVANTE", caso_irrelevante)]:
    rel  = AnswerRelevancyMetric(threshold=0.7)
    fai  = FaithfulnessMetric(threshold=0.7)
    hal  = HallucinationMetric(threshold=0.5)
    rel.measure(caso); fai.measure(caso); hal.measure(caso)
    print(f"\n=== {nombre} ===")
    print(f"  relevancy={rel.score:.2f}  faithfulness={fai.score:.2f}  hallucination={hal.score:.2f}")

/home/vicente/anaconda3/envs/agents/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


=== OK ===
  relevancy=1.00  faithfulness=1.00  hallucination=0.00



=== ALUCINA ===
  relevancy=0.50  faithfulness=0.00  hallucination=1.00



=== IRRELEVANTE ===
  relevancy=0.00  faithfulness=1.00  hallucination=0.00


---
## Ejercicio 5b (bonus) — RAGAS para RAG

RAGAS evalúa pipelines RAG. Diagnostica si el problema está en el **retriever** (context precision/recall) o en el **generador** (faithfulness).

In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference

muestra = SingleTurnSample(
    user_input="Que es un LLM?",
    response="Un LLM es un modelo de lenguaje grande entrenado con mucho texto.",
    retrieved_contexts=["Un LLM es una red neuronal entrenada con grandes corpus de texto."],
    reference="Un modelo de lenguaje grande (LLM) es una red neuronal entrenada con texto.",
)
dataset = EvaluationDataset(samples=[muestra])

resultado = evaluate(
    dataset,
    metrics=[Faithfulness(), ResponseRelevancy(), LLMContextPrecisionWithReference()],
)
print(resultado)

---
## Ejercicio 6 — LangSmith (dataset + evaluator + experimento)

Crea un dataset, define un evaluator (LLM-as-judge de correctness) y compara dos prompts como experimentos. Requiere `LANGSMITH_API_KEY` y `LANGSMITH_TRACING=true`.

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate as ls_evaluate

client = Client()
DATASET = "capitales-demo"

# 1) Crear dataset (idempotente)
if not client.has_dataset(dataset_name=DATASET):
    ds = client.create_dataset(DATASET)
    client.create_examples(
        inputs=[{"q": "Capital de Peru?"}, {"q": "Capital de Francia?"}],
        outputs=[{"a": "Lima"}, {"a": "Paris"}],
        dataset_id=ds.id,
    )

In [ ]:
# 2) Evaluator: correctness (comparacion simple, case-insensitive)
def correctness(run, example):
    pred = (run.outputs or {}).get("a", "").strip().lower()
    gold = (example.outputs or {}).get("a", "").strip().lower()
    return {"key": "correctness", "score": int(gold in pred)}

# 3) Dos versiones de la app (dos prompts) para comparar
def app_v1(inputs):
    return {"a": ask("gpt", f"Responde solo el nombre. {inputs['q']}")}

def app_v2(inputs):
    return {"a": ask("gpt", f"Eres un geografo. {inputs['q']} Responde con una sola palabra.")}

for nombre, app in [("prompt-v1", app_v1), ("prompt-v2", app_v2)]:
    ls_evaluate(app, data=DATASET, evaluators=[correctness],
                experiment_prefix=nombre)
    print(f"Experimento '{nombre}' enviado. Revisa el dashboard de LangSmith.")

---
## Ejercicio 7 — Elegir un modelo según el caso de uso

Consulta los leaderboards en vivo y usa la tabla para decidir. Aquí automatizamos una **recomendación razonada** pidiéndosela a un LLM con datos comparativos como contexto.

- Artificial Analysis (Agentic Index): https://artificialanalysis.ai/?intelligence=agentic-index
- Vellum: https://www.vellum.ai/llm-leaderboard#compare

In [ ]:
# Tabla ilustrativa (reemplaza con datos reales del leaderboard)
tabla = """
modelo            | mmlu_pro | gpqa | agentic | precio_$Mtok | tokens_s
Gemini 3.5 Pro    |   88     |  84  |   alto  |   medio      |  alto
GPT-5.5           |   89     |  85  |   alto  |   alto       |  medio
Claude Opus 4.8   |   90     |  86  |  muy alto|  alto       |  medio
Claude Sonnet 4.6 |   86     |  80  |   alto  |   bajo       |  alto
DeepSeek V4       |   85     |  79  |   medio |  muy bajo    |  alto
"""

escenarios = [
    "Chatbot de soporte de alto volumen (prioriza costo y velocidad).",
    "Asistente de investigacion cientifica (prioriza GPQA/razonamiento).",
    "Agente de codigo (prioriza capacidad agentica / SWE-bench).",
]

prompt = (f"Dada esta tabla de modelos:\n{tabla}\n\n"
          f"Para cada escenario recomienda UN modelo y justifica en 1 linea "
          f"con metricas concretas:\n- " + "\n- ".join(escenarios))
display(Markdown(ask("gpt", prompt)))

---
## Ejercicio (agentes) — Tool Correctness y Task Completion

Construimos un agente con 2 herramientas y evaluamos si **usó las tools correctas** y si **completó la tarea**.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def get_weather(city: str) -> str:
    """Devuelve el clima (mock) de una ciudad en grados Celsius."""
    return f"{city}: 20C, despejado"

@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convierte grados Celsius a Fahrenheit."""
    return celsius * 9 / 5 + 32

agent = create_agent(
    init_chat_model(MODELS["gpt"]),
    tools=[get_weather, celsius_to_fahrenheit],
)

pregunta = "Que clima hace en Lima y convierte esa temperatura a Fahrenheit?"
resultado = agent.invoke({"messages": [("user", pregunta)]})

# Extraer las tools que el agente realmente llamo, en orden
tools_llamadas = []
for m in resultado["messages"]:
    for tc in getattr(m, "tool_calls", []) or []:
        tools_llamadas.append(tc["name"])

respuesta_final = resultado["messages"][-1].content
print("Tools llamadas:", tools_llamadas)
print("Respuesta:", respuesta_final)

In [ ]:
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import ToolCorrectnessMetric, TaskCompletionMetric

caso_agente = LLMTestCase(
    input=pregunta,
    actual_output=respuesta_final,
    tools_called=[ToolCall(name=n) for n in tools_llamadas],
    expected_tools=[ToolCall(name="get_weather"),
                    ToolCall(name="celsius_to_fahrenheit")],
)

tool_metric = ToolCorrectnessMetric()
tool_metric.measure(caso_agente)
print(f"Tool Correctness: {tool_metric.score:.2f} -> {tool_metric.reason}")

# Reto: pregunta que solo necesita UNA tool. Llama de mas el agente?
p2 = "Convierte 100 grados Celsius a Fahrenheit."
res2 = agent.invoke({"messages": [("user", p2)]})
tools2 = [tc["name"] for m in res2["messages"]
          for tc in getattr(m, "tool_calls", []) or []]
print("\nTools usadas en tarea simple:", tools2, "(idealmente solo celsius_to_fahrenheit)")

---
### Resumen

| Necesito... | Uso... |
|---|---|
| Comparar modelos base | benchmarks, leaderboards (Ej. 7) |
| Métrica clásica vs. semántica | BLEU + embeddings (Ej. 1) |
| Detectar memorización | variantes de ítems (Ej. 2) |
| Comparar respuestas | LLM-as-a-judge (Ej. 4) |
| Evaluar mi RAG | DeepEval / RAGAS (Ej. 5) |
| Monitorear / comparar prompts | LangSmith (Ej. 6) |
| Evaluar agentes | Tool Correctness / Task Completion |